In [ ]:
VISION_PROMPT = """Ты — ассистент по оцифровке научных графиков. По изображению определи все кривые и выпиши их в текстовом виде.

Порядок работы:
1. Определи масштаб осей X и Y по числовым меткам. Если метки только на краях — интерполируй. Если сетки нет — оценивай положение кривой относительно осей.
2. Определи, сколько кривых на графике: одна или семейство.
3. Для каждой кривой:
   - укажи значение третьего параметра, подписанного рядом (например: "param: T=300"). Если кривая одна — "param: none".
   - выпиши не менее 6 точек в виде пар "x, y", равномерно по всей длине кривой.

Правила:
- Используй вычесленный масштаб осей, не перепроверяй постоянно.
- Значения x и y — только числа (int или float), без единиц измерения, без текста.
- Первая и последняя точки каждой кривой — это её крайние точки на графике. Не добавляй дополнительных точек на минимуме/максимуме оси X, если кривая там не проходит.
- Игнорируй отдельные точки, не лежащие на кривых.
- Игнорируй подписи-комментарии (например, пометки про максимумы, пояснения к участкам). Из текста выделяй ТОЛЬКО значения варьирующегося параметра.
- Если сомневаешься, горизонтальная ли линия — считай горизонтальной.
- Если линия выглядит ломаной — считай её ломаной и обязательно включай в набор все точки излома (вершины, где меняется наклон). Внутри прямых участков между изломами точки можно не добавлять, но сами вершины — обязательны.
- Не пиши вступление, не рассуждай, не перепроверяй значения. Сразу начинай вывод с первого блока CURVE. Одна итерация, без самокоррекции.

Формат вывода — простой текст, БЕЗ markdown, БЕЗ таблиц, БЕЗ JSON:

CURVE
param: <значение или none>
x, y
x, y
...

CURVE
param: ...
x, y
...

Цель — набор точек, достаточно хорошо согласующийся с изображением.
"""

In [ ]:
import base64
import json
import re
import sys
from pathlib import Path

import pandas as pd
import requests

from IPython.display import display  # работает и в Jupyter, и в JupyterLab

OLLAMA_URL   = "http://192.168.0.5:11434/api/chat"
VISION_MODEL = "qwen3-vl:32b"
TEXT_MODEL   = "qwen3:32b"           # или "gemma3:27b"
IMAGE_PATH   = "graph.png"
OUTPUT_CSV   = "digitized_graph.csv"
TIMEOUT      = 1800                  # для стриминга лучше побольше
TAIL_LEN     = 2000                  # сколько последних символов держать в экране

# ============================================================
#  Стриминговый вызов: в реальном времени показывает thinking+content
# ============================================================


In [25]:
import base64
import json
import re
import sys
import traceback
from pathlib import Path

import pandas as pd
import requests

# --- Конфигурация ---
IMAGES_DIR      = Path("pictures")       # папка с *.png
OUTPUT_DIR      = Path("digitized")    # сюда пишем csv и debug-файлы
SKIP_IF_CSV_EXISTS = False             # True — пропускать уже обработанные (есть .csv)

OLLAMA_URL      = "http://192.168.0.5:11434/api/chat"
VISION_MODEL    = "qwen3.8:27b"        # было "qwen3-vl:32b"
TEXT_MODEL      = "qwen3.8:27b"        # или "gemma3:27b", "gemma4:31b"
TIMEOUT         = 1800
NUM_PREDICT     = 6144
NUM_CTX         = 60000
TAIL_LEN        = 2000


# --- Промпт для vision-модели (стадия 1) ---
VISION_PROMPT = """Ты — ассистент по оцифровке научных графиков. По изображению определи все кривые и выпиши их в текстовом виде.

Порядок работы:
1. Определи масштаб осей X и Y по числовым меткам. Если метки только на краях — интерполируй. Если сетки нет — оценивай положение кривой относительно осей.
2. Определи, сколько кривых на графике: одна или семейство.
3. Для каждой кривой:
   - укажи значение третьего параметра, подписанного рядом (например: "param: T=300"). Если кривая одна — "param: none".
   - выпиши не менее 6 точек в виде пар "x, y", равномерно по всей длине кривой.

Правила:
- Значения x и y — только числа (int или float), без единиц измерения, без текста.
- Первая и последняя точки каждой кривой — это её крайние точки на графике. Не добавляй дополнительных точек на минимуме/максимуме оси X, если кривая там не проходит.
- Игнорируй отдельные точки, не лежащие на кривых.
- Игнорируй подписи-комментарии (например, пометки про максимумы, пояснения к участкам). Из текста выделяй ТОЛЬКО значения варьирующегося параметра.
- Если сомневаешься, горизонтальная ли линия — считай горизонтальной.
- Если линия выглядит ломаной — считай её ломаной и обязательно включай в набор все точки излома (вершины, где меняется наклон). Внутри прямых участков между изломами точки можно не добавлять, но сами вершины — обязательны.

Формат вывода — простой текст, БЕЗ markdown, БЕЗ таблиц, БЕЗ JSON:

CURVE
param: <значение или none>
x, y
x, y
...

CURVE
param: ...
x, y
...

Цель — набор точек, достаточно хорошо согласующийся с изображением.
"""

# --- Промпт для текстовой модели (стадия 2) ---
TEXT_PROMPT_PREFIX = """Тебе дан сырой текстовый вывод другой модели, которая оцифровывала график с картинки.
Твоя задача — извлечь из этого текста данные и вернуть СТРОГО валидный JSON по схеме:

{
  "is_family": true | false,
  "curves": [
    {
      "param_value": "значение параметра как в тексте, или null",
      "points": [{"x": число, "y": число}, ...]
    }
  ]
}

Правила:
- Если в тексте один блок CURVE или явно одна кривая — is_family=false, param_value=null.
- Если блоков несколько — is_family=true, для каждого блока свой объект с param_value из строки "param:".
- x и y — обязательно ЧИСЛА (float/int), а не строки. Если в тексте числа с запятой как десятичным разделителем — замени на точку.
- Не придумывай точки, которых нет в тексте. Если для какой-то кривой точек нет — points: [].
- Ответ — только JSON, без markdown, без пояснений.

=== СЫРОЙ ТЕКСТ НАЧАЛО ===

"""

TEXT_PROMPT_SUFFIX = """

=== СЫРОЙ ТЕКСТ КОНЕЦ ===
"""


# ---------- IPython / fallback ----------
try:
    from IPython.display import display, clear_output
    _HAS_IPY = True
except Exception:
    _HAS_IPY = False


def _make_display():
    """Возвращает функцию-обновлятор. В Jupyter — display_id, иначе — print."""
    if _HAS_IPY:
        handle = display({"text/plain": ""}, raw=True, display_id=True)

        def updater(text: str):
            try:
                handle.update({"text/plain": text}, raw=True)
            except Exception:
                pass
        return updater

    state = {"last": 0}

    def updater(text: str):
        if len(text) - state["last"] > 400:
            print(text[-TAIL_LEN:], flush=True)
            state["last"] = len(text)
    return updater


def maybe_clear():
    if _HAS_IPY:
        clear_output(wait=True)


# ---------- HTTP ----------
def encode_image_to_base64(path: Path) -> str:
    return base64.b64encode(path.read_bytes()).decode("utf-8")


def call_chat(model: str, messages: list, json_mode: bool = False,
              timeout: int = TIMEOUT, num_predict: int = NUM_PREDICT,
              num_ctx: int = NUM_CTX, think=True, label: str = "") -> dict:
    payload = {
        "model": model,
        "messages": messages,
        "stream": True,
        "think": think,
        "options": {
            "temperature": 0.0,
            "num_predict": num_predict,
            "num_ctx": num_ctx,
        },
    }
    if json_mode:
        payload["format"] = "json"

    print(f"→ POST {OLLAMA_URL} model={model} json_mode={json_mode} {label}")

    show_think   = _make_display()
    show_content = _make_display()

    full_content, full_thinking = "", ""

    with requests.post(OLLAMA_URL, json=payload, timeout=timeout,
                       headers={"Content-Type": "application/json"},
                       stream=True) as r:
        if r.status_code != 200:
            print(f"HTTP {r.status_code}: {r.text[:1000]}", file=sys.stderr)
            r.raise_for_status()

        for line in r.iter_lines():
            if not line:
                continue
            try:
                chunk = json.loads(line)
            except json.JSONDecodeError:
                continue

            msg = chunk.get("message", {}) or {}
            d_think   = msg.get("thinking", "") or ""
            d_content = msg.get("content", "") or ""

            if d_think:
                full_thinking += d_think
                show_think("💭 …" + full_thinking[-TAIL_LEN:])
            if d_content:
                full_content += d_content
                show_content("📝 …" + full_content[-TAIL_LEN:])
            if chunk.get("done"):
                break

    show_think("💭 (готово, " + str(len(full_thinking)) + " симв.)")
    show_content("📝 (готово, " + str(len(full_content)) + " симв.)")

    return {"message": {"content": full_content, "thinking": full_thinking}}


def message_text(resp: dict) -> str:
    msg = resp.get("message", {}) or {}
    content = msg.get("content", "") or ""
    thinking = msg.get("thinking", "") or ""
    return content, thinking


# ---------- Парсинг JSON ----------
def iter_json_candidates(text: str):
    t = text.strip()
    t = re.sub(r"^```(?:json)?\s*", "", t, flags=re.IGNORECASE)
    t = re.sub(r"\s*```$", "", t, flags=re.IGNORECASE)

    a, b = t.find("{"), t.rfind("}")
    if a != -1 and b > a:
        try:
            yield json.loads(t[a:b + 1])
        except json.JSONDecodeError as e:
            print(f"[extract] цельный JSON не сработал: {e}")

    objs = []
    for m in re.finditer(r"\{", t):
        depth, start = 0, m.start()
        for i in range(start, len(t)):
            if t[i] == "{":
                depth += 1
            elif t[i] == "}":
                depth -= 1
                if depth == 0:
                    try:
                        objs.append(json.loads(t[start:i + 1]))
                    except json.JSONDecodeError:
                        pass
                    break
    objs.sort(key=lambda o: -len(json.dumps(o)))
    yield from objs


def find_curves(obj):
    if isinstance(obj, dict):
        if isinstance(obj.get("curves"), list):
            return obj["curves"]
        for v in obj.values():
            r = find_curves(v)
            if r is not None:
                return r
    elif isinstance(obj, list):
        if obj and isinstance(obj[0], dict) and ("points" in obj[0] or "param_value" in obj[0]):
            return obj
        if obj and isinstance(obj[0], dict) and "x" in obj[0] and "y" in obj[0]:
            return [{"param_value": None, "points": obj}]
    return None


def curves_to_rows(curves):
    rows = []
    for c in curves or []:
        param = c.get("param_value") if isinstance(c, dict) else None
        pts = c.get("points", []) if isinstance(c, dict) else []
        for p in pts:
            if isinstance(p, dict):
                x, y = p.get("x"), p.get("y")
            elif isinstance(p, (list, tuple)) and len(p) >= 2:
                x, y = p[0], p[1]
            else:
                continue
            rows.append({"param": param, "x": x, "y": y})
    return rows


# ---------- Обработка одного изображения ----------
def process_image(img_path: Path, out_dir: Path) -> dict:
    stem = img_path.stem
    csv_path    = out_dir / f"{stem}.csv"
    stage1_path = out_dir / f"{stem}.stage1.txt"
    stage2_path = out_dir / f"{stem}.stage2.txt"

    if SKIP_IF_CSV_EXISTS and csv_path.exists():
        return {"file": img_path.name, "status": "skipped",
                "curves": None, "rows": None, "note": "csv exists"}

    img_b64 = encode_image_to_base64(img_path)

    # --- Стадия 1: vision читает картинку ---
    resp1 = call_chat(
        VISION_MODEL,
        [{"role": "user", "content": VISION_PROMPT, "images": [img_b64]}],
        json_mode=False,
        num_predict=4096,
        think=True,
        label="[stage1/vision]",
    )
    c1, t1 = message_text(resp1)
    stage1_text = (c1 + "\n" + t1).strip()
    stage1_path.write_text(stage1_text, encoding="utf-8")
    print(f"стадия 1: content={len(c1)} thinking={len(t1)} -> {stage1_path.name}")

    if not stage1_text:
        return {"file": img_path.name, "status": "error",
                "curves": None, "rows": None, "note": "empty stage1"}

    # --- Стадия 2: текстовая модель -> JSON ---
    stage2_user = TEXT_PROMPT_PREFIX + stage1_text + TEXT_PROMPT_SUFFIX
    resp2 = call_chat(
        TEXT_MODEL,
        [{"role": "user", "content": stage2_user}],
        json_mode=True,
        num_predict=6144,
        label="[stage2/json]",
    )
    c2, t2 = message_text(resp2)
    stage2_text = (c2 + "\n" + t2).strip()
    stage2_path.write_text(stage2_text, encoding="utf-8")
    print(f"стадия 2: content={len(c2)} thinking={len(t2)} -> {stage2_path.name}")

    # --- Парсим JSON ---
    curves, parsed = None, None
    for cand in iter_json_candidates(stage2_text):
        cs = find_curves(cand)
        if cs is not None:
            curves, parsed = cs, cand
            break
    if curves is None:
        return {"file": img_path.name, "status": "error",
                "curves": None, "rows": None, "note": "no curves in stage2"}

    rows = curves_to_rows(curves)
    if not rows:
        return {"file": img_path.name, "status": "error",
                "curves": len(curves), "rows": 0, "note": "no x/y points"}

    df = pd.DataFrame(rows)
    df["x"] = pd.to_numeric(df["x"], errors="coerce")
    df["y"] = pd.to_numeric(df["y"], errors="coerce")
    df = df.dropna(subset=["x", "y"]).reset_index(drop=True)

    if isinstance(parsed, dict) and parsed.get("is_family") is False:
        df = df.drop(columns=["param"])

    df.to_csv(csv_path, index=False)
    return {"file": img_path.name, "status": "ok",
            "curves": len(curves), "rows": len(df), "note": ""}


# ---------- main ----------
def main():
    if not IMAGES_DIR.exists():
        print(f"❌ Нет папки {IMAGES_DIR}")
        return None

    images = sorted(IMAGES_DIR.glob("*.png"))
    if not images:
        print(f"❌ В {IMAGES_DIR} нет *.png")
        return None

    OUTPUT_DIR.mkdir(exist_ok=True)
    print(f"Найдено {len(images)} png. Результаты → {OUTPUT_DIR}/")

    results = []
    for i, img_path in enumerate(images, 1):
        maybe_clear()
        print(f"=== [{i}/{len(images)}] {img_path.name} ===")
        try:
            r = process_image(img_path, OUTPUT_DIR)
        except Exception as e:
            traceback.print_exc()
            r = {"file": img_path.name, "status": "error",
                 "curves": None, "rows": None, "note": str(e)}
        print(f"   → status={r['status']} curves={r['curves']} rows={r['rows']} {r['note']}")
        results.append(r)

    # --- итоговая сводка ---
    maybe_clear()
    summary = pd.DataFrame(results)
    n_ok = (summary["status"] == "ok").sum()
    print(f"Готово: {n_ok}/{len(results)} ok")
    print(summary.to_string(index=False))

    # сводку сохраняем рядом с результатами
    summary_csv = OUTPUT_DIR / "_summary.csv"
    summary.to_csv(summary_csv, index=False)
    print(f"Сводка сохранена в {summary_csv}")
    return summary


if __name__ == "__main__":
    main()

Готово: 5/5 ok
            file status  curves  rows note
KPD_GTU_NGTU.png     ok       1     9     
  NGTU_TCOMP.png     ok       6    66     
N_COMP_N_GTU.png     ok       1     8     
PICOMP_GCOMP.png     ok       1     7     
   qGTU_NGTU.png     ok       1    11     
Сводка сохранена в digitized/_summary.csv
